# Haiku kNN Retrieval

**Project Name:** Haiku (renamed from Haiku)

## Purpose
- Publication-ready notebook for reproducible training/evaluation.

## Notes
- Paths and checkpoints may be environment-specific.
- Run cells top-to-bottom and set your config paths first.


In [ ]:
import sys
from pathlib import Path

HAIKU_ROOT = Path('/home/yancui/Haiku')
if str(HAIKU_ROOT / 'src') not in sys.path:

from haiku.notebook_utils import setup_notebook, seed_everything
setup_notebook(project_root='/home/yancui/Haiku')
seed_everything(42)


In [ ]:
import hydra
from omegaconf import DictConfig, OmegaConf
import os
### This is needed to get torch running on the gpu1 queue, if using other GPUs that dont have MIG divisions, dont need to set this var before importing torch
#os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5"
import torch
from torch.utils.data import DataLoader, RandomSampler
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.distributed.nn.functional as dist_fn
from torch import distributed as dist
from torchvision import transforms
from tqdm import tqdm
import wandb
import pickle
from os.path import join
import pandas as pd
import numpy as np
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
import random
import json

import sys





from models import Haiku, MarkerEmbedding
from data import custom_collate_fn_trimodal, TrimodalDatasetViT
from utils import PairwiseCLIPLoss, OneVersusAllLoss, PerChannelSelfStandardization, CustomGaussianBlurTorch
from datetime import timedelta, datetime
import csv

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Load yaml config
cfg = OmegaConf.load('/home/yancui/Haiku/src/configs/config.yaml')


In [ ]:
import json
import pandas as pd



#sample_dict = json.load(open('/home/yancui/Haiku/overlap_samples.json'))
sample_dict = json.load(open('/home/yancui/Haiku/src/training/overlap_samples_final.json'))
# sample_ids = sample_ids = pd.read_csv('/project/zhihuanglab/jleiby/codex_clip/sample_ids_with_text.csv', header=None)[0].tolist()
sample_ids = list(sample_dict.keys())


with open('/home/yancui/OmicsAnnotator/test_regions.txt', 'r') as f:
    test_ids = [line.strip() for line in f if line.strip()]

#sample_ids = test_ids

holdout_list = pd.read_csv('/home/yancui/tier2_acquisition_ids_huanglab (3).csv')['ACQUISITION_ID'].tolist()

overlap_holdout_list = list(set(holdout_list) & set(sample_ids))

new_holdout_list = list(json.load(open('/home/yancui/Haiku/overlap_samples_new.json')).keys())

sample_ids = list(set(test_ids + list(set(overlap_holdout_list) - set(new_holdout_list))))

ref_ids = sorted(sample_ids)

In [ ]:

import os
from tqdm import tqdm

region_metadata_dir = "/data/enable_data/region_metadata"

# First, read all region_metadata CSVs into a dict: {region_id: df}
region_metadata = {}
metadata_files = [fname for fname in os.listdir(region_metadata_dir) if fname.endswith('.metadata.csv')]
print(f"Reading {len(metadata_files)} region metadata CSVs...")
for fname in tqdm(metadata_files, desc="Reading region metadata", total=len(metadata_files)):
    region_id = fname.split('.')[0]
    try:
        df = pd.read_csv(os.path.join(region_metadata_dir, fname))
        region_metadata[region_id] = df
    except Exception as e:
        print(f"Error reading {fname}: {e}")


In [ ]:
import torch

he_embedding = torch.load('/home/yancui/Haiku/res_embdding_126/he_embedding.pt')
codex_embedding = torch.load('/home/yancui/Haiku/res_embdding_126/codex_embedding.pt')
region_label = torch.load('/home/yancui/Haiku/res_embdding_126/region_label.pt')
virtual_codex_embedding = torch.load('/home/yancui/Haiku/res_embdding_126/virtual_codex_embedding.pt')
text_embedding = torch.load('/home/yancui/Haiku/res_embdding_126/text_embedding.pt')
musk_he_embedding = torch.load('/home/yancui/Haiku/res_embdding_126/baseline_musk_he_embedding.pt')
musk_text_embedding = torch.load('/home/yancui/Haiku/res_embdding_126/baseline_musk_text_embedding.pt')
musk_codex_embedding = torch.load('/home/yancui/Haiku/res_embdding_126/baseline_musk_codex_embedding.pt')


In [ ]:
ref_ids = sorted(sample_ids)

metadata_dict_values = {}
metadata_dict_ids = {}

keys = ['tissue_type', 'diagnosis', 'disease', 'Pathology diagnosis', 'stage', 'survival', 'type', 'survival_status', 'grade']

for key in keys:
    metadata_dict_values[key] = []
    metadata_dict_ids[key] = []

print(f"Processing {len(region_label)} samples for metadata lookup...")
for i, sample in tqdm(enumerate(region_label), total=len(region_label), desc="Processing metadata"):
    #patch_id = sample['patch_id']
    #print(sample)
    region_id = ref_ids[sample].split('_')[0]
    df = region_metadata.get(region_id, None)
    if df is not None:
        for key in keys:
            if key in df['FEATURE_NAME'].values:
                if (df[df['FEATURE_NAME'] == key]['FEATURE_VALUE'].values[0] == 'nan') or (df[df['FEATURE_NAME'] == key]['FEATURE_VALUE'].values[0] == 'unknown') or (df[df['FEATURE_NAME'] == key]['FEATURE_VALUE'].values[0] == 'Unknown') or (df[df['FEATURE_NAME'] == key]['FEATURE_VALUE'].values[0] == np.nan):
                    continue
                else:
                    metadata_dict_values[key].append(df[df['FEATURE_NAME'] == key]['FEATURE_VALUE'].values[0])
                    metadata_dict_ids[key].append(i)

In [ ]:
category_map = {}

category_map['type'] = {
    'normal': 'Normal',
    'nat': 'Normal',
    'at': 'Normal',
    'AT': 'Normal',
    "NAT": 'Normal',
    'hyperplasia': 'Benign/Precancerous',
    'malignant': 'Primary Tumor',
    'tumor primary': 'Primary Tumor',
    'Tumor Primary': 'Primary Tumor',
    'Maglignant': 'Primary Tumor',
    'metastasis': 'Metastatic Tumor',
    'nan': None,
    '-': None,
    '*': None
}

category_map['grade'] = {
    '1': 'G1',
    '1--2': 'G1',
    'g1': 'G1',
    '2': 'G2',
    '2--3': 'G2',
    'g2': 'G2',
    '3': 'G3',
    'g3': 'G3',
    'nan': None,
    '-': None,
    '*': None
}


In [ ]:
keys = ['tissue_type', 'grade', 'tnm', 'type']


def clean_labels(labels, map):
    """
    Map various survival status labels to 'alive' or 'death'.
    """
    mapped = []
    for x in labels:
        if x in map:
            mapped.append(map[x])
        else:
            mapped.append(x)

    return np.array(mapped)


for key in metadata_dict_values:
    filtered_values = []
    filtered_ids = []
    for val, idx in zip(metadata_dict_values[key], metadata_dict_ids[key]):
        if (
            val is not None
            and str(val).lower() != 'nan'
            and str(val).lower() != 'unknown'
            and str(val).lower() != '-'
            and not (isinstance(val, float) and np.isnan(val))
        ):
            filtered_values.append(val)
            filtered_ids.append(idx)
    metadata_dict_values[key] = filtered_values
    metadata_dict_ids[key] = filtered_ids


for key in metadata_dict_values:
    if key in category_map:
        filtered_values = clean_labels(metadata_dict_values[key], category_map[key])
        metadata_dict_ids[key] = np.array(metadata_dict_ids[key])[filtered_values != None]
        metadata_dict_values[key] = np.array(filtered_values)[filtered_values != None]


In [ ]:
import torch
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F


# -----------------------------
# KNN classification utilities
# -----------------------------
def _unique_preserve_order(seq):
    seen, out = set(), []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def _labels_to_indices(labels, label_vocab=None):
    """
    Map arbitrary labels to contiguous indices.
    If label_vocab not provided, infer from labels (order-preserving).
    Returns:
        idx_tensor (LongTensor [N]), label_vocab (list)
    """
    if label_vocab is None:
        label_vocab = _unique_preserve_order(labels)
    lut = {lab: i for i, lab in enumerate(label_vocab)}
    idx = torch.tensor([lut[l] for l in labels], dtype=torch.long)
    return idx, label_vocab

def _compute_confusion(y_true_idx, y_pred_idx, n_class):
    """
    Build confusion matrix C where C[i,j] = count(true=i, pred=j)
    """
    C = torch.zeros((n_class, n_class), dtype=torch.long)
    for t, p in zip(y_true_idx.tolist(), y_pred_idx.tolist()):
        C[t, p] += 1
    return C

def _f1_from_confusion(C):
    """
    Given confusion matrix C (n_class x n_class), compute:
      - accuracy
      - F1-micro
      - F1-macro
    Note: For single-label multiclass, micro-F1 == accuracy.
    """
    n_class = C.size(0)
    tp = torch.diag(C).to(torch.float64)
    fp = C.sum(dim=0).to(torch.float64) - tp
    fn = C.sum(dim=1).to(torch.float64) - tp
    support = C.sum(dim=1).to(torch.float64)

    # Per-class precision/recall/F1 (handle zero-division -> 0)
    precision_c = torch.where(tp + fp > 0, tp / (tp + fp), torch.zeros_like(tp))
    recall_c    = torch.where(tp + fn > 0, tp / (tp + fn), torch.zeros_like(tp))
    f1_c_denom  = precision_c + recall_c
    f1_c        = torch.where(f1_c_denom > 0, 2 * precision_c * recall_c / f1_c_denom, torch.zeros_like(f1_c_denom))

    # Macro over classes that appear in y_true (support > 0)
    mask_present = support > 0
    if mask_present.any():
        f1_macro = f1_c[mask_present].mean().item()
    else:
        f1_macro = 0.0

    total = C.sum().item()
    correct = tp.sum().item()
    accuracy = correct / total if total > 0 else 0.0

    # Micro-F1 == accuracy for multiclass single-label
    f1_micro = accuracy

    return accuracy, f1_micro, f1_macro


# ------------------------------------------
# Main: KNN classifier over embedding space
# ------------------------------------------
def compute_knn_metrics_minibatch(
    query_embeddings,
    gallery_embeddings,
    query_labels,
    gallery_labels,
    top_ks=(1, 5, 10, 20, 50),
    batch_size=256,
    device='cuda',
    similarity='cosine',         # 'cosine' or 'dot'
    vote='weighted',             # 'uniform' or 'weighted' (by similarity)
    normalize_embeddings=True,   # L2-normalize before sim
    verbose=True
):
    """
    Evaluate KNN classification using gallery as the reference set and queries as test points.

    Args:
        query_embeddings: Tensor [Nq, D]
        gallery_embeddings: Tensor [Ng, D]
        query_labels: list/array length Nq (class labels)
        gallery_labels: list/array length Ng (class labels)
        top_ks: iterable of K values to evaluate
        batch_size: batch size for queries
        device: 'cuda' or 'cpu'
        similarity: 'cosine' (recommended) or 'dot'
        vote: 'uniform' (majority vote) or 'weighted' (sum of similarities per class)
        normalize_embeddings: if True, L2-normalize embeddings before similarity
        verbose: print progress and summary

    Returns:
        metrics: dict with keys:
          - 'accuracy@K', 'F1_micro@K', 'F1_macro@K'
          for each K in top_ks
    """
    assert len(query_labels) == query_embeddings.size(0)
    assert len(gallery_labels) == gallery_embeddings.size(0)

    Nq = query_embeddings.size(0)
    Ng = gallery_embeddings.size(0)
    top_ks = tuple(sorted(set(int(k) for k in top_ks if k > 0)))
    max_k = max(top_ks)

    # Prepare tensors
    q = query_embeddings.to(device)
    g = gallery_embeddings.to(device)
    if normalize_embeddings:
        q = F.normalize(q, dim=1)
        g = F.normalize(g, dim=1)

    # Label vocab (indices)
    # Use union so predictions that pick a class absent in y_true still map cleanly.
    label_vocab = _unique_preserve_order(list(gallery_labels) + list(query_labels))
    y_true_idx, _ = _labels_to_indices(list(query_labels), label_vocab)
    g_lab_idx, _  = _labels_to_indices(list(gallery_labels), label_vocab)
    y_true_idx = y_true_idx.to(torch.long)
    g_lab_idx  = g_lab_idx.to(torch.long)

    # Collect predictions per K
    preds_per_k = {k: [] for k in top_ks}

    if verbose:
        print(f"\n[INFO] KNN classification over embeddings:")
        print(f" - Queries:  {Nq}")
        print(f" - Gallery:  {Ng}")
        print(f" - K values: {top_ks}")
        print(f" - Similarity: {similarity} | Vote: {vote} | Normalize: {normalize_embeddings}")
        print(f" - Device: {device} | Batch size: {batch_size}")

    num_batches = (Nq + batch_size - 1) // batch_size
    iterator = range(0, Nq, batch_size)
    if verbose:
        iterator = tqdm(iterator, total=num_batches, desc="Evaluating KNN")

    with torch.no_grad():
        for start in iterator:
            end = min(start + batch_size, Nq)
            batch_q = q[start:end]  # [B, D]

            # Similarity matrix [B, Ng]
            if similarity == 'cosine' or (similarity == 'dot' and normalize_embeddings):
                sim = batch_q @ g.t()
            elif similarity == 'dot':
                sim = torch.matmul(batch_q, g.t())
            else:
                raise ValueError("similarity must be 'cosine' or 'dot'.")

            # Top max_k neighbors
            top_sim, top_idx = torch.topk(sim, k=min(max_k, Ng), dim=1, largest=True, sorted=True)
            top_idx = top_idx.to('cpu')  # [B, K*]
            top_labels = g_lab_idx[top_idx]  # [B, K*]

            # For each requested K, do voting
            for K in top_ks:
                if K > top_idx.size(1):
                    # not enough gallery points, fall back to available
                    K_use = top_idx.size(1)
                else:
                    K_use = K

                neigh_labels = top_labels[:, :K_use]        # [B, K_use]
                neigh_sims   = top_sim[:, :K_use]           # [B, K_use]

                if vote == 'uniform':
                    # Majority vote -> count occurrences per class
                    # Build bincount per row
                    preds = []
                    for i in range(neigh_labels.size(0)):
                        counts = torch.bincount(
                            neigh_labels[i],
                            minlength=len(label_vocab)
                        )
                        # Tie-breaker: pick class with highest total similarity if tie,
                        # else smallest index
                        ties = (counts == counts.max()).nonzero(as_tuple=True)[0]
                        if len(ties) == 1:
                            preds.append(ties.item())
                        else:
                            # similarity tie-break
                            sim_sum = torch.zeros(len(label_vocab), device=neigh_labels.device)
                            sim_sum.scatter_add_(0, neigh_labels[i], neigh_sims[i])
                            sim_sum_ties = sim_sum[ties]
                            best = ties[sim_sum_ties.argmax()].item()
                            preds.append(best)
                    preds = torch.tensor(preds, dtype=torch.long)
                elif vote == 'weighted':
                    # Sum similarities per class, pick argmax
                    preds = []
                    for i in range(neigh_labels.size(0)):
                        scores = torch.zeros(len(label_vocab), device=neigh_labels.device)
                        scores.scatter_add_(0, neigh_labels[i], neigh_sims[i])
                        preds.append(scores.argmax().item())
                    preds = torch.tensor(preds, dtype=torch.long)
                else:
                    raise ValueError("vote must be 'uniform' or 'weighted'.")

                preds_per_k[K].append(preds.cpu())

    # Concatenate predictions per K and compute metrics
    metrics = {}
    y_true_all = y_true_idx  # [Nq]
    for K in top_ks:
        y_pred_all = torch.cat(preds_per_k[K], dim=0)  # [Nq]
        C = _compute_confusion(y_true_all, y_pred_all, n_class=len(label_vocab))

        acc, f1_micro, f1_macro = _f1_from_confusion(C)
        metrics[f"accuracy@{K}"]  = acc
        metrics[f"F1_micro@{K}"]  = f1_micro
        metrics[f"F1_macro@{K}"]  = f1_macro

    if verbose:
        print("\n=== Final KNN Classification Metrics ===")
        for K in top_ks:
            print(f"K={K:>3} | Acc: {metrics[f'accuracy@{K}']:.4f} | "
                  f"F1-micro: {metrics[f'F1_micro@{K}']:.4f} | F1-macro: {metrics[f'F1_macro@{K}']:.4f}")

    return metrics


# ---------------------------------------------------
# Random baseline: shuffle gallery labels once & eval
# ---------------------------------------------------
def compute_knn_metrics_random_once(
    query_embeddings,
    gallery_embeddings,
    query_labels,
    gallery_labels,
    top_ks=(1, 5, 10, 20, 50),
    batch_size=256,
    device='cuda',
    seed=42,
    **kwargs
):
    """
    Build a random baseline by shuffling gallery labels (embeddings unchanged).
    """
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(gallery_labels))
    shuffled_gallery_labels = [gallery_labels[i] for i in perm]
    return compute_knn_metrics_minibatch(
        query_embeddings=query_embeddings,
        gallery_embeddings=gallery_embeddings,
        query_labels=query_labels,
        gallery_labels=shuffled_gallery_labels,
        top_ks=top_ks,
        batch_size=batch_size,
        device=device,
        verbose=False,
        **kwargs
    )


# --------------------------
# Plotting: bar comparison
# --------------------------
def plot_knn_metrics_barplot(
    ours_metrics,
    baseline_metrics,
    top_ks=(1, 5, 10, 20, 50),
    metric_names=("accuracy", "F1_macro", "F1_micro"),
    ours_label="Ours",
    baseline_label="Random Baseline"
):
    """
    Grouped bars comparing KNN classification metrics across K.

    Args:
        ours_metrics: dict from compute_knn_metrics_minibatch
        baseline_metrics: dict from compute_knn_metrics_random_once
        top_ks: iterable of K to show
        metric_names: tuple/list of metrics among {"accuracy","F1_macro","F1_micro"}
        ours_label, baseline_label: legend labels
    """
    import numpy as np
    import matplotlib.pyplot as plt

    metric_names = list(metric_names)
    n_metrics = len(metric_names)
    ks = list(top_ks)
    x = np.arange(len(ks))
    width = 0.36

    fig, axes = plt.subplots(1, n_metrics, figsize=(4*n_metrics, 4), sharey=False)
    if n_metrics == 1:
        axes = [axes]

    def _title(m):
        if m == "accuracy":
            return "Accuracy"
        elif m == "F1_macro":
            return "F1-macro"
        elif m == "F1_micro":
            return "F1-micro"
        return m

    for i, m in enumerate(metric_names):
        ours_vals = []
        base_vals = []
        for k in ks:
            ours_vals.append(ours_metrics.get(f"{m}@{k}", np.nan))
            base_vals.append(baseline_metrics.get(f"{m}@{k}", np.nan))
        ax = axes[i]
        ax.bar(x - width/2, ours_vals, width, label=ours_label)
        ax.bar(x + width/2, base_vals, width, label=baseline_label)
        ax.set_xticks(x)
        ax.set_xticklabels([str(k) for k in ks])
        ax.set_xlabel("K (neighbors)")
        ax.set_title(_title(m))
        ax.set_ylim(0, 1)
        if i == 0:
            ax.set_ylabel("Score")
        ax.legend()
        for idx, v in enumerate(ours_vals):
            if np.isfinite(v):
                ax.text(idx - width/2, min(v + 0.01, 0.995), f"{v:.2f}", ha='center', va='bottom', fontsize=9)
        for idx, v in enumerate(base_vals):
            if np.isfinite(v):
                ax.text(idx + width/2, min(v + 0.01, 0.995), f"{v:.2f}", ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.show()


# --------------------------
# Example usage (pseudo)
# ----------------------
#
# ----
res_retrieval_results = {'Codex-to-HE': {}, 'HE-to-Codex': {}, 'Text-to-Codex': {}}

keys = ['tissue_type', 'grade', 'type']

for key in keys:
    q_labels = metadata_dict_values[key]
    g_labels = metadata_dict_values[key]
    ids = metadata_dict_ids[key]
    res_retrieval_results['Codex-to-HE'][key] = {}

    ours = compute_knn_metrics_minibatch(
        query_embeddings=codex_embedding.to('cpu')[ids],          # torch.Tensor [Nq,D]
        gallery_embeddings=he_embedding.to('cpu')[ids],        # torch.Tensor [Ng,D]
        query_labels=q_labels,       # list length Nq
        gallery_labels=g_labels,     # list length Ng
        top_ks=(1,5,10,20,50),
        batch_size=256,
        device='cpu',
        similarity='cosine',
        vote='weighted',
        normalize_embeddings=True,
    )
    #
    print(musk_codex_embedding.to('cpu')[ids].shape, musk_he_embedding.to('cpu')[ids].shape, len(q_labels), len(g_labels))
    random = compute_knn_metrics_random_once(
            query_embeddings=musk_codex_embedding.to('cpu')[ids],
            gallery_embeddings=musk_he_embedding.to('cpu')[ids],
            query_labels=q_labels,
            gallery_labels=g_labels,
            top_ks=(1,5,10,20,50),
            batch_size=256,
            device='cpu',
            seed=42,
            similarity='cosine',
            vote='weighted',
            normalize_embeddings=True,
        )

    musk = compute_knn_metrics_minibatch(
        query_embeddings=musk_codex_embedding.to('cpu')[ids],          # torch.Tensor [Nq,D]
        gallery_embeddings=musk_he_embedding.to('cpu')[ids],        # torch.Tensor [Ng,D]
        query_labels=q_labels,       # list length Nq
        gallery_labels=g_labels,     # list length Ng
        top_ks=(1,5,10,20,50),
        batch_size=256,
        device='cpu',
        similarity='cosine',
        vote='weighted',
        normalize_embeddings=True,
    )


    res_retrieval_results['Codex-to-HE'][key]['Ours'] = ours
    res_retrieval_results['Codex-to-HE'][key]['Random'] = random
    res_retrieval_results['Codex-to-HE'][key]['Musk'] = musk


res_retrieval_results['Codex-to-HE']['Region'] = {}

q_labels = region_label.squeeze(1).numpy().astype(str)
g_labels = region_label.squeeze(1).numpy().astype(str)

ours = compute_knn_metrics_minibatch(
    query_embeddings=codex_embedding.to('cpu'),          # torch.Tensor [Nq,D]
    gallery_embeddings=he_embedding.to('cpu'),        # torch.Tensor [Ng,D]
    query_labels=q_labels,       # list length Nq
    gallery_labels=g_labels,     # list length Ng
    top_ks=(1,5,10,20,50),
    batch_size=256,
    device='cpu',
    similarity='cosine',
    vote='weighted',
    normalize_embeddings=True,
)
#
musk = compute_knn_metrics_minibatch(
    query_embeddings=musk_codex_embedding.to('cpu'),          # torch.Tensor [Nq,D]
    gallery_embeddings=musk_he_embedding.to('cpu'),        # torch.Tensor [Ng,D]
    query_labels=q_labels,       # list length Nq
    gallery_labels=g_labels,     # list length Ng
    top_ks=(1,5,10,20,50),
    batch_size=256,
    device='cpu',
    similarity='cosine',
    vote='weighted',
    normalize_embeddings=True,
)

random = compute_knn_metrics_random_once(
        query_embeddings=musk_codex_embedding.to('cpu'),
        gallery_embeddings=musk_he_embedding.to('cpu'),
        query_labels=q_labels,
        gallery_labels=g_labels,
        top_ks=(1,5,10,20,50),
        batch_size=256,
        device='cpu',
        seed=42,
        similarity='cosine',
        vote='weighted',
        normalize_embeddings=True,
    )

res_retrieval_results['Codex-to-HE']['Region']['Ours'] = ours
res_retrieval_results['Codex-to-HE']['Region']['Musk'] = musk
res_retrieval_results['Codex-to-HE']['Region']['Random'] = random

In [ ]:
for key in keys:
    q_labels = metadata_dict_values[key]
    g_labels = metadata_dict_values[key]
    ids = metadata_dict_ids[key]
    res_retrieval_results['HE-to-Codex'][key] = {}

    ours = compute_knn_metrics_minibatch(
        query_embeddings=he_embedding.to('cpu')[ids],          # torch.Tensor [Nq,D]
        gallery_embeddings=codex_embedding.to('cpu')[ids],        # torch.Tensor [Ng,D]
        query_labels=q_labels,       # list length Nq
        gallery_labels=g_labels,     # list length Ng
        top_ks=(1,5,10,20,50),
        batch_size=256,
        device='cpu',
        similarity='cosine',
        vote='weighted',
        normalize_embeddings=True,
    )
    #
    random = compute_knn_metrics_random_once(
            query_embeddings=musk_he_embedding.to('cpu')[ids],
            gallery_embeddings=musk_codex_embedding.to('cpu')[ids],
            query_labels=q_labels,
            gallery_labels=g_labels,
            top_ks=(1,5,10,20,50),
            batch_size=256,
            device='cpu',
            seed=42,
            similarity='cosine',
            vote='weighted',
            normalize_embeddings=True,
        )

    musk = compute_knn_metrics_minibatch(
        query_embeddings=musk_he_embedding.to('cpu')[ids],          # torch.Tensor [Nq,D]
        gallery_embeddings=musk_codex_embedding.to('cpu')[ids],        # torch.Tensor [Ng,D]
        query_labels=q_labels,       # list length Nq
        gallery_labels=g_labels,     # list length Ng
        top_ks=(1,5,10,20,50),
        batch_size=256,
        device='cpu',
        similarity='cosine',
        vote='weighted',
        normalize_embeddings=True,
    )
    res_retrieval_results['HE-to-Codex'][key]['Musk'] = musk
    res_retrieval_results['HE-to-Codex'][key]['Ours'] = ours
    res_retrieval_results['HE-to-Codex'][key]['Random'] = random
    plot_knn_metrics_barplot(ours, musk, top_ks=(1,5,10,20,50))



res_retrieval_results['HE-to-Codex']['Region'] = {}

q_labels = region_label.squeeze(1).numpy().astype(str)
g_labels = region_label.squeeze(1).numpy().astype(str)

ours = compute_knn_metrics_minibatch(
    query_embeddings=he_embedding.to('cpu'),          # torch.Tensor [Nq,D]
    gallery_embeddings=codex_embedding.to('cpu'),        # torch.Tensor [Ng,D]
    query_labels=q_labels,       # list length Nq
    gallery_labels=g_labels,     # list length Ng
    top_ks=(1,5,10,20,50),
    batch_size=256,
    device='cpu',
    similarity='cosine',
    vote='weighted',
    normalize_embeddings=True,
)
#
random = compute_knn_metrics_random_once(
    query_embeddings=musk_he_embedding.to('cpu'),
    gallery_embeddings=musk_codex_embedding.to('cpu'),
    query_labels=q_labels,
    gallery_labels=g_labels,
    top_ks=(1,5,10,20,50),
    batch_size=256,
    device='cpu',
    seed=42,
    similarity='cosine',
    vote='weighted',
    normalize_embeddings=True,
)

musk = compute_knn_metrics_minibatch(
    query_embeddings=musk_he_embedding.to('cpu'),
    gallery_embeddings=musk_codex_embedding.to('cpu'),
    query_labels=q_labels,
    gallery_labels=g_labels,
    top_ks=(1,5,10,20,50),
    batch_size=256,
    device='cpu',
    similarity='cosine',
    vote='weighted',
    normalize_embeddings=True,
)

res_retrieval_results['HE-to-Codex']['Region']['Ours'] = ours
res_retrieval_results['HE-to-Codex']['Region']['Musk'] = musk
res_retrieval_results['HE-to-Codex']['Region']['Random'] = random

plot_knn_metrics_barplot(ours, musk, top_ks=(1,5,10,20,50))

In [ ]:
for key in keys:
    q_labels = metadata_dict_values[key]
    g_labels = metadata_dict_values[key]
    ids = metadata_dict_ids[key]
    res_retrieval_results['Text-to-Codex'][key] = {}

    ours = compute_knn_metrics_minibatch(
        query_embeddings=text_embedding.to('cpu')[ids],          # torch.Tensor [Nq,D]
        gallery_embeddings=codex_embedding.to('cpu')[ids],        # torch.Tensor [Ng,D]
        query_labels=q_labels,       # list length Nq
        gallery_labels=g_labels,     # list length Ng
        top_ks=(1,5,10,20,50),
        batch_size=256,
        device='cpu',
        similarity='cosine',
        vote='weighted',
        normalize_embeddings=True,
    )
    #
    random = compute_knn_metrics_random_once(
            query_embeddings=musk_text_embedding.to('cpu')[ids],
            gallery_embeddings=musk_codex_embedding.to('cpu')[ids],
            query_labels=q_labels,
            gallery_labels=g_labels,
            top_ks=(1,5,10,20,50),
            batch_size=256,
            device='cpu',
            seed=42,
            similarity='cosine',
            vote='weighted',
            normalize_embeddings=True,
        )

    musk = compute_knn_metrics_minibatch(
        query_embeddings=musk_text_embedding.to('cpu')[ids],
        gallery_embeddings=musk_codex_embedding.to('cpu')[ids],
        query_labels=q_labels,
        gallery_labels=g_labels,
        top_ks=(1,5,10,20,50),
        batch_size=256,
        device='cpu',
        similarity='cosine',
        vote='weighted',
        normalize_embeddings=True,
    )


    res_retrieval_results['Text-to-Codex'][key]['Ours'] = ours
    res_retrieval_results['Text-to-Codex'][key]['Musk'] = musk
    res_retrieval_results['Text-to-Codex'][key]['Random'] = random
    plot_knn_metrics_barplot(ours, musk, top_ks=(1,5,10,20,50))


res_retrieval_results['Text-to-Codex']['Region'] = {}

q_labels = region_label.squeeze(1).numpy().astype(str)
g_labels = region_label.squeeze(1).numpy().astype(str)

ours = compute_knn_metrics_minibatch(
    query_embeddings=text_embedding.to('cpu'),          # torch.Tensor [Nq,D]
    gallery_embeddings=codex_embedding.to('cpu'),        # torch.Tensor [Ng,D]
    query_labels=q_labels,       # list length Nq
    gallery_labels=g_labels,     # list length Ng
    top_ks=(1,5,10,20,50),
    batch_size=256,
    device='cpu',
    similarity='cosine',
    vote='weighted',
    normalize_embeddings=True,
)
#
random = compute_knn_metrics_random_once(
    query_embeddings=musk_text_embedding.to('cpu'),
    gallery_embeddings=musk_codex_embedding.to('cpu'),
    query_labels=q_labels,
    gallery_labels=g_labels,
    top_ks=(1,5,10,20,50),
    batch_size=256,
    device='cpu',
    seed=42,
    similarity='cosine',
    vote='weighted',
    normalize_embeddings=True,
)

musk = compute_knn_metrics_minibatch(
    query_embeddings=musk_text_embedding.to('cpu'),
    gallery_embeddings=musk_codex_embedding.to('cpu'),
    query_labels=q_labels,
    gallery_labels=g_labels,
    top_ks=(1,5,10,20,50),
    batch_size=256,
    device='cpu',
    similarity='cosine',
    vote='weighted',
    normalize_embeddings=False,
)


res_retrieval_results['Text-to-Codex']['Region']['Ours'] = ours
res_retrieval_results['Text-to-Codex']['Region']['Musk'] = musk
res_retrieval_results['Text-to-Codex']['Region']['Random'] = random

plot_knn_metrics_barplot(ours, musk, top_ks=(1,5,10,20,50))

In [ ]:
import json

with open('retrieval_results_knn_111.json', 'w') as f:


In [ ]:
with open('retrieval_results_knn_111.json', 'r') as f:
    retrival_results = json.load(f)

In [ ]:
retrival_results

In [ ]:
retrival_results['HE-to-Codex']

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# -------- settings --------
JSON_PATH = "retrieval_results_knn_111.json"   # change if stored elsewhere
METHODS   = ["Ours", "Musk", "Random"]               # case-insensitive; "MUSK" also works
METRIC    = "F1_macro@1"                   # only this metric is plotted

# Keep text editable in Illustrator for SVG output
plt.rcParams['font.family'] = 'Arial'    # use Arial font
plt.rcParams['font.size'] = 12           # set large font size    # make text bold
plt.rcParams['svg.fonttype'] = 'none'    # don't convert text to path
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['font.weight'] = 'bold' # embed as TrueType for AI compatibility
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['axes.labelweight'] = 'bold'# bold axis labels
plt.rcParams['axes.titleweight'] = 'bold'# bold title

# ---------- load ----------
with open(JSON_PATH, "r") as f:
    results = json.load(f)

def resolve_method_key(any_label_dict, target_name):
    """Find 'target_name' key in dict ignoring case (e.g., 'MUSK' vs 'Musk')."""
    if target_name in any_label_dict:
        return target_name
    t = target_name.lower()
    for k in any_label_dict.keys():
        if k.lower() == t:
            return k
    return None

def barplot_for_task(task_name, labels_dict):
    labels = list(labels_dict.keys())
    n = len(labels)
    if n == 0:
        return

    x = np.arange(n)  # label positions
    width = 0.33      # bar width

    num_methods = len(METHODS)
    # Figure width: 2.2 inches per method
    fig_width = max(6, 2.2 * num_methods + 3)
    fig_height = 6.5 if n < 6 else 7
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    for i, meth in enumerate(METHODS):
        # Pick one label’s dict to resolve key variant once
        first_label_dict = next(iter(labels_dict.values()))
        mkey = resolve_method_key(first_label_dict, meth)
        if mkey is None:
            continue

        vals = np.array([
            labels_dict[lbl].get(mkey, {}).get(METRIC, np.nan) for lbl in labels
        ], dtype=float)

        bars = ax.bar(x + i*width, vals, width, label=meth, alpha=0.90)

        # Annotate bars
        for xpos, val in zip(x + i*width, vals):
            if np.isfinite(val):
                ax.text(xpos, min(1.0, val + 0.02), f"{val:.3f}",
                        ha="center", va="bottom", fontsize=9)

    # Appearance
    ax.set_xticks(x + width*(num_methods-1)/2)
    ax.set_xticklabels(labels, fontsize=11, rotation=14)
    ax.set_ylabel(METRIC, fontsize=12)
    ax.set_title(f"{task_name} — {METRIC}", fontsize=13, pad=10)
    ax.set_ylim(0, 1.0)
    ax.legend(loc="upper right", frameon=True)
    fig.tight_layout()

    # Save SVG/PNG
    stem = f"{task_name.replace(' ', '_')}_bar_{METRIC.replace('@','at')}"
    plt.show()

# -------- generate all tasks --------
# (Adjust ylim of the figure to make bars more compact.)

def get_bar_ylim(labels_dict, methods=METHODS, metric=METRIC, pad=0.04):
    vals = []
    for meth in methods:
        # Pick one label’s dict to resolve key variant once
        if not labels_dict:
            continue
        first_label_dict = next(iter(labels_dict.values()))
        mkey = resolve_method_key(first_label_dict, meth)
        if mkey is None:
            continue
        for lbl in labels_dict:
            v = labels_dict[lbl].get(mkey, {}).get(metric, np.nan)
            if np.isfinite(v):
                vals.append(v)
    if not vals:
        return 0.0, 1.0
    mn, mx = min(vals), max(vals)
    # shrink range but ensure at least a small pad left/right
    if mn == mx:
        return max(0, mn - pad), min(1, mx + pad)
    gap = mx - mn
    ylim_lo = max(0, mn - pad * gap)
    ylim_hi = min(1.0, mx + 2*pad * gap)
    # Ensure small minimum span
    if ylim_hi - ylim_lo < 0.12:
        mid = (ylim_hi + ylim_lo)/2
        span = 0.06
        ylim_lo = max(0, mid - span)
        ylim_hi = min(1, mid + span)
    return ylim_lo, ylim_hi

# Wrap to override ax.set_ylim line on the fly:
def barplot_for_task_compactylim(task_name, labels_dict):
    labels = list(labels_dict.keys())
    n = len(labels)
    if n == 0:
        return

    num_methods = len(METHODS)

    # ---------- spacing controls ----------
    GROUP_SPACING = 1.35   # >1 => more separation between label groups
    x = np.arange(n) * GROUP_SPACING

    width = 0.26           # bar width
    intra_gap = 0.0        # IMPORTANT: 0 => methods touch (tied together)

    group_total = num_methods * width + (num_methods - 1) * intra_gap

    # Center offsets so the whole method-block is centered at x
    offsets = []
    left = -group_total / 2 + width / 2
    for i in range(num_methods):
        offsets.append(left + i * (width + intra_gap))

    # ---------- font sizes ----------
    XTICK_FONTSIZE = 16
    YTICK_FONTSIZE = 16
    AXIS_LABEL_FONTSIZE = 16
    TITLE_FONTSIZE = 16
    ANNO_FONTSIZE = 14
    LEGEND_FONTSIZE = 13

    fig_width = max(6, 2.2 * num_methods + 3)
    fig_height = 6.5 if n < 6 else 7
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    for i, meth in enumerate(METHODS):
        if not labels_dict:
            continue
        first_label_dict = next(iter(labels_dict.values()))
        mkey = resolve_method_key(first_label_dict, meth)
        if mkey is None:
            continue

        vals = np.array(
            [labels_dict[lbl].get(mkey, {}).get(METRIC, np.nan) for lbl in labels],
            dtype=float
        )

        xpos = x + offsets[i]
        ax.bar(
            xpos, vals, width,
            label=meth,
            alpha=0.90,
            edgecolor="black",
            linewidth=1
        )

        # annotate
        for xx, val in zip(xpos, vals):
            if np.isfinite(val):
                ax.text(
                    xx, val + 0.02, f"{val:.3f}",
                    ha="center", va="bottom",
                    fontsize=ANNO_FONTSIZE
                )

    # x ticks at group centers
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=XTICK_FONTSIZE, rotation=14)

    ax.set_ylabel(METRIC, fontsize=AXIS_LABEL_FONTSIZE)
    ax.set_title(f"{task_name} — {METRIC}", fontsize=TITLE_FONTSIZE, pad=10)

    ax.tick_params(axis="x", labelsize=XTICK_FONTSIZE, width=1.2, length=5)
    ax.tick_params(axis="y", labelsize=YTICK_FONTSIZE, width=1.2, length=5)

    # compact ylim
    ylim_lo, ylim_hi = get_bar_ylim(labels_dict)
    ax.set_ylim(ylim_lo, ylim_hi)

    # remove top/right spines
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.2)
    ax.spines["bottom"].set_linewidth(1.2)

    ax.legend(loc="upper right", frameon=True, fontsize=LEGEND_FONTSIZE)
    fig.tight_layout()

    stem = f"{task_name.replace(' ', '_')}_bar_{METRIC.replace('@','at')}"
    plt.show()


for task, label_block in results.items():
    barplot_for_task_compactylim(task, label_block)
